# Encoder and Random Forest and larger datasets

So far we have been limiting ourselves to single subsets of the data. A specfic instrument, ionization mode and what not. In this case I want to take several datasets, train them with different encoders then use a union of the outputs to train a much larger Random forest (ideally balaning that RF).

## Import packages

In [28]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

## Globally used Functions

In [29]:
# Spectrum string to dataframe function
def spectrum_string_to_dataframe(df, spectrum_col, smiles_col):
    """
    Converts a DataFrame with a spectrum column (string of 'x:y' pairs) into a matrix
    where columns are unique x values, rows are spectra (even for duplicate SMILES), and values are y (intensity).
    The index will match the original DataFrame.
    """
    # Collect all unique x values (m/z)
    x_values_set = set()
    spectra_list = []
    for idx, row in df.iterrows():
        spectrum = row[spectrum_col]
        pairs = spectrum.split()
        xys = []
        for pair in pairs:
            try:
                x, y = pair.split(":") # Split into x and y
                #x = float(x.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                #y = float(y.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                xys.append((x, y))
                x_values_set.add(x)
            except Exception:
                continue
        spectra_list.append((row[smiles_col], dict(xys)))
    x_values = sorted(x_values_set) # Sort the x values to maintain order
    
    # Build the matrix
    matrix = []
    smiles_list = []
    for smiles, xy_dict in spectra_list:
        row = [xy_dict.get(x, 0.0) for x in x_values]
        matrix.append(row)
        smiles_list.append(smiles)
    df_matrix = pd.DataFrame(matrix, columns=[x for x in x_values]) # columns=[f"mz_{x}" for x in x_values]) to make stings
    df_matrix.insert(0, smiles_col, smiles_list)
    df_matrix.index = df.index  # preserve original row order/index
    return df_matrix

In [30]:
# Binning functions
# Binning the spectra data 
def bin_spectra_by_integer_mz(df):
    """
    Bins the spectra data by rounding m/z (column names) to the nearest integer,
    then sums intensities for duplicate integer bins.
    Assumes first column is SMILES, rest are m/z columns (floats).
    """
    smiles_col = df.columns[0]
    spectra = df.iloc[:, 1:]
    # Map each column to its integer bin
    int_mz = [int(round(float(c))) for c in spectra.columns]
    spectra.columns = int_mz
    # Group columns by integer m/z and sum
    binned = spectra.groupby(level=0, axis=1).sum()
    # Add the SMILES column back
    binned.insert(0, smiles_col, df[smiles_col])
    return binned

# Fill in the missing integer columns 
def fill_missing_integer_columns(df):
    """
    Ensures all integer columns from 1 to the maximum are present in the DataFrame.
    Missing columns are filled with zeros.
    Assumes the first column is the label (e.g., SMILES).
    """
    # Get the integer columns (skip the first column)
    int_cols = [col for col in df.columns[1:] if isinstance(col, int)]
    #min_col = min(int_cols)
    max_col = max(int_cols)
    all_int_cols = list(range(1, max_col + 1))
    # Find missing columns
    missing_cols = set(all_int_cols) - set(int_cols)
    # Add missing columns with zeros
    for col in missing_cols:
        df[col] = 0.0
    # Reorder columns: first column, then sorted integer columns
    ordered_cols = [df.columns[0]] + sorted(all_int_cols)
    df = df[ordered_cols]
    return df

## Data Processing: Upload, Edits, and Split

In [ ]:
# The 5/20 dataset with rat based toxicity data
df3 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data3.csv")
print(df3.shape)
df3.head()


# Uniformity of ionization model labels
print(df3["Ionization_Mode"].unique())
df3["Ionization_Mode"] = df3["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df3["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df3 = df3[df3["Ionization_Mode"] != "'NaN'"]
print(df3["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df3
df3 = df3.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df3["SMILES_spectra"] = df3["SMILES_spectra"].str.replace("'", "")

# Data split based on the 'Group' column
df3_QQpos = df3[df3['Group'] == 'Q-Orbitrap-positive'] # 1307
df3_QQneg = df3[df3['Group'] == 'Q-Orbitrap-negative'] # 756
df3_QTOFpos = df3[df3['Group'] == 'Q-TOF-positive'] # 736  
df3_LTQOpos = df3[df3['Group'] == 'LTQ-Orbitrap-positive'] # 481 

# With these there may not be enough data to train a model
df3_QQQpos = df3[df3['Group'] == 'QQQ-positive'] # 253
df3_QTOFneg = df3[df3['Group'] == 'Q-TOF-negative'] # 188
df3_QQQneg = df3[df3['Group'] == 'QQQ-negative'] # 85
df3_Opos = df3[df3['Group'] == 'Other-positive'] # 71
df3_LTQOneg = df3[df3['Group'] == 'LTQ-Orbitrap-negative'] # 63
df3_LTQpos = df3[df3['Group'] == 'LTQ-positive'] # 19
df3_QQQnan = df3[df3['Group'] == 'QQQ-nan'] # 18
df3_Oneg = df3[df3['Group'] == 'Other-negative'] # 13
df3_LTQneg = df3[df3['Group'] == 'LTQ-negative'] # 11

(4001, 16)
["'positive'" "'negative'" "'Positive'" "'NaN'"]
["'positive'" "'negative'" "'NaN'"]
["'positive'" "'negative'"]


In [43]:
df3_QTOFpos.head()

,SMILES_spectra,CAS,Molecular_Formula,Total_Exact_Mass,Precursor_m/z,Precursor_Type,Spectrum,Ionization_Mode,Instrument_Type,Instrument_Name,Collision_Energy,SMILES_tox_vals,Response_Modifier,Response,Response_Unit,Group


## Data Processing: Binning

In [36]:
df3_QQpos_matrix = spectrum_string_to_dataframe(df3_QQpos,'Spectrum', 'SMILES_spectra')
df3_QQpos_spectra = df3_QQpos_matrix
df3_QQneg_matrix = spectrum_string_to_dataframe(df3_QQneg,'Spectrum', 'SMILES_spectra')
df3_QQneg_spectra = df3_QQneg_matrix
df3_QTOFpos_matrix = spectrum_string_to_dataframe(df3_QTOFpos,'Spectrum', 'SMILES_spectra')
df3_QTOFpos_spectra = df3_QTOFpos_matrix
df3_LTQOpos_matrix = spectrum_string_to_dataframe(df3_LTQOpos,'Spectrum', 'SMILES_spectra')
df3_LTQOpos_spectra = df3_LTQOpos_matrix

In [39]:
# Processing of the spectra to enable binning to work (consider changing this to a function)

# Conversion of m/z values to floats, keeping the first column (SMIELS) as is
# For df3_QQpos_spectra
cols = df3_QQpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QQpos_spectra.columns = new_cols

# For df3_QQneg_spectra
cols = df3_QQneg_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QQneg_spectra.columns = new_cols

# For df3_QTOFpos_spectra
cols = df3_QTOFpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_QTOFpos_spectra.columns = new_cols

# For df3_LTQOpos_spectra
cols = df3_LTQOpos_spectra.columns.tolist()
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df3_LTQOpos_spectra.columns = new_cols


# Convert all elements except the first column to float
df3_QQpos_spectra.iloc[:, 1:] = df3_QQpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_QQneg_spectra.iloc[:, 1:] = df3_QQneg_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_QTOFpos_spectra.iloc[:, 1:] = df3_QTOFpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df3_LTQOpos_spectra.iloc[:, 1:] = df3_LTQOpos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

all_float = all(isinstance(c, float) for c in df3_QQneg_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_QQpos_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_QTOFpos_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df3_LTQOpos_spectra.columns[1:])
print("All columns are float:", all_float)

# Check if every element is a float
spectra = df3_QQpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_QQneg_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_QTOFpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)
spectra = df3_LTQOpos_spectra.iloc[:, 1:]
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)

# Sort columns by float value, keep the first column (SMILES) first
cols = df3_QQpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QQpos_spectra = df3_QQpos_spectra[sorted_cols]
cols = df3_QQneg_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QQneg_spectra = df3_QQneg_spectra[sorted_cols]
cols = df3_QTOFpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_QTOFpos_spectra = df3_QTOFpos_spectra[sorted_cols]
cols = df3_LTQOpos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df3_LTQOpos_spectra = df3_LTQOpos_spectra[sorted_cols]

All columns are float: True
All columns are float: True
All columns are float: True
All columns are float: True
All elements are float: True
All elements are float: True
All elements are float: True
All elements are float: True


In [42]:
df3_QTOFpos_spectra.head()

,SMILES_spectra


In [41]:
# Use the binning function 
# On df3_QQpos_spectra
binned_df3_QQpos_spectra = bin_spectra_by_integer_mz(df3_QQpos_spectra)
binned_df3_QQpos_spectra_filled = fill_missing_integer_columns(binned_df3_QQpos_spectra)

# On df3_QQpos_spectra
binned_df3_QQneg_spectra = bin_spectra_by_integer_mz(df3_QQneg_spectra)
binned_df3_QQneg_spectra_filled = fill_missing_integer_columns(binned_df3_QQneg_spectra)

# On df3_QQpos_spectra
binned_df3_QTOFpos_spectra = bin_spectra_by_integer_mz(df3_QTOFpos_spectra)
binned_df3_QTOFpos_spectra_filled = fill_missing_integer_columns(binned_df3_QTOFpos_spectra)

# On df3_LTQOpos_spectra
binned_df3_LTQOpos_spectra = bin_spectra_by_integer_mz(df3_LTQOpos_spectra)
binned_df3_LTQOpos_spectra_filled = fill_missing_integer_columns(binned_df3_LTQOpos_spectra)

/tmp/ipykernel_115872/339873635.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  binned.insert(0, smiles_col, df[smiles_col])
/tmp/ipykernel_115872/339873635.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = 0.0
/tmp/ipykernel_115872/339873635.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

ValueError: max() arg is an empty sequence

In [ ]:
print(binned_df3_QQpos_spectra_filled.shape)
print(binned_df3_QQneg_spectra_filled.shape)
print(binned_df3_QTOFpos_spectra_filled.shape)
print(binned_df3_LTQOpos_spectra_filled.shape)